# CLM-0.4 Preview
Product-facing Preview run with resumable checkpoints, public telemetry, visualizations, and GitHub result publication.

In [ ]:
from pathlib import Path
import os, subprocess, sys, json, torch
WORK = Path('/kaggle/working')
ROOT = WORK / 'mini-cells'
if not ROOT.exists():
    subprocess.run(['git','clone','https://github.com/ArcheLabs/mini-cells.git',str(ROOT)], check=True)
else:
    subprocess.run(['git','switch','main'], cwd=ROOT, check=True)
    subprocess.run(['git','pull','--ff-only','origin','main'], cwd=ROOT, check=True)
os.chdir(ROOT)
print('HEAD', subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip())
print('GPUs', torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() >= 2, 'Preview Kaggle run expects T4 x2'

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-e',str(ROOT)+'[lm]'], check=True)
sys.path.insert(0, str(ROOT / 'src')) if str(ROOT / 'src') not in sys.path else None
print('MiniCells ready')

In [ ]:
DATA = WORK / 'clm-0.4-preview-data'
REVISION = 'f54c09fd23315a6f9c86f9dc80f725de7d8f9c64'
if not (DATA / 'asset-summary.json').is_file():
    subprocess.run([sys.executable, str(ROOT/'scripts/research/prepare_clm_0_4_preview_data.py'), '--dataset-revision', REVISION, '--out', str(DATA)], cwd=ROOT, check=True)
assets = json.loads((DATA/'asset-summary.json').read_text())
print(json.dumps(assets, indent=2, sort_keys=True))

In [ ]:
OUT = WORK / 'clm-0.4-preview'
cmd = [sys.executable, str(ROOT/'scripts/research/run.py'), 'clm-0.4-preview', '--data-dir', str(DATA), '--out', str(OUT), '--device', 'cuda', '--devices', 'cuda:0,cuda:1', '--max-transactions', '192']
subprocess.run(cmd, cwd=ROOT, check=True)
print(json.dumps(json.loads((OUT/'decision.json').read_text()), indent=2))

In [ ]:
subprocess.run([sys.executable, str(ROOT/'scripts/research/report.py'), 'clm-0.4-preview', '--results', str(OUT)], cwd=ROOT, check=True)
print((OUT/'PUBLIC_METRICS.md').read_text())
print((OUT/'RESULTS.md').read_text())

In [ ]:
PUSH_RESULTS = True
if PUSH_RESULTS:
    subprocess.run([sys.executable, str(ROOT/'scripts/research/publish.py'), 'clm-0.4-preview', '--results', str(OUT), '--push'], cwd=ROOT, check=True)
else:
    print('Set PUSH_RESULTS=True to publish curated Preview telemetry.')